<a href="https://colab.research.google.com/github/ipavlopoulos/diachronic-greek-letterforms/blob/main/notebooks/resnet_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet-18 Training Workflow

This notebook trains the ResNet-18 baseline/variant on the released Hell-Char cliplets using the same split, lacuna-style augmentation, and similarity-weighted supervised contrastive learning utilities used by the lightweight CNN workflow. It saves `best_resnet_supcon_model.pth` and then evaluates the saved model on the held-out Hell-Char test split.

The released repository does not include a ResNet checkpoint, so rerunning this notebook is the reproducible path for generating it. Exact scores can vary slightly across hardware and library versions unless all random seeds and deterministic kernels are fixed.


## Setup

The notebook can be run from the repository root, from the `notebooks/` folder, or in Colab. A GPU is recommended for ResNet training.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/ipavlopoulos/diachronic-greek-letterforms.git"
REPO_DIR = Path("/content/diachronic-greek-letterforms")

if IN_COLAB and not Path("source.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
elif not Path("source.py").exists() and Path("..").joinpath("source.py").exists():
    os.chdir("..")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("Working directory:", Path.cwd())

import random

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from source import (
    ImageDatasetAugmented,
    RandomLacunae,
    ResNetClassifier,
    custom_similarity_matrix,
    evaluate,
    train_cnn2d,
    tta_transform,
)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load Hell-Char

Labels are read from the cliplet filenames, matching the CNN training notebook. Samples labelled `Unknown` are excluded from supervised training and evaluation.


In [ ]:
image_folder = Path("data/hellchar/cliplets")
metadata_path = Path("data/hellchar/hellchar.csv")

image_files = sorted(
    path for path in image_folder.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

metadata = pd.read_csv(metadata_path)
data = pd.DataFrame({"path": image_files, "filename": [path.name for path in image_files]})
data["letter"] = data.filename.apply(lambda x: x.split("_")[0])
data["TM"] = data.filename.apply(lambda x: int(x.split("_")[1]))
data["number"] = data.filename.apply(lambda x: x.split("_")[2].split(".")[0])

data["year"] = data.TM.apply(lambda x: metadata.loc[metadata["TM"] == x]["Year ante quem"].values[0])
data["region"] = data.TM.apply(lambda x: metadata.loc[metadata["TM"] == x]["Production Nome (supposed)"].values[0])

data.head()


## Preprocess And Split

Images are resized to `64 x 64` grayscale arrays. The split follows the CNN workflow: 20% held out for testing, then 10% of the remaining labelled samples held out for validation, both stratified by letter.


In [ ]:
def preprocess_image_2d(image_path, size=(64, 64), otsu=False):
    img = Image.open(image_path).convert("L")
    img_np = np.array(img)

    if otsu:
        _, img_np = cv2.threshold(img_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        img_np = 255 - img_np

    img_resized = cv2.resize(img_np, size, interpolation=cv2.INTER_AREA)
    return img_resized.astype(np.float32) / 255.0

image_data_2d = np.array([preprocess_image_2d(path) for path in data.path])

known_data = data[data["letter"] != "Unknown"].copy()
known_indices = known_data.index.tolist()

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(known_data["letter"])
num_classes_2d = len(label_encoder.classes_)

train_indices_2d, test_indices_2d, y_train_encoded_2d, y_test_encoded_2d = train_test_split(
    known_indices,
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

train_indices_2d, val_indices_2d, y_train_encoded_2d, y_val_encoded_2d = train_test_split(
    train_indices_2d,
    y_train_encoded_2d,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_encoded_2d,
)

X_train_2d = image_data_2d[train_indices_2d]
y_train_2d = y_train_encoded_2d
X_val_2d = image_data_2d[val_indices_2d]
y_val_2d = y_val_encoded_2d
X_test_2d = image_data_2d[test_indices_2d]
y_test_2d = y_test_encoded_2d

print(f"Classes: {num_classes_2d}")
print(f"Train/val/test: {len(X_train_2d)}/{len(X_val_2d)}/{len(X_test_2d)}")


## Dataloaders

The training loader applies the same augmentation family as the CNN workflow, including lacuna-style erasure. Validation and test loaders use only tensor conversion and normalization.


In [ ]:
data_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    RandomLacunae(num_lacunae=(0, 2), size_range=(0.02, 0.12), p=0.5, v=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

batch_size = 32
train_loader_2d_aug = DataLoader(
    ImageDatasetAugmented(X_train_2d, y_train_2d, transform=data_transform),
    batch_size=batch_size,
    shuffle=True,
)
val_loader_2d = DataLoader(
    ImageDatasetAugmented(X_val_2d, y_val_2d, transform=test_transform),
    batch_size=batch_size,
    shuffle=False,
)
test_loader_2d = DataLoader(
    ImageDatasetAugmented(X_test_2d, y_test_2d, transform=test_transform),
    batch_size=batch_size,
    shuffle=False,
)


## Train ResNet-18

This cell fine-tunes a grayscale ResNet-18 and saves the best validation-loss checkpoint as `best_resnet_supcon_model.pth`. Reduce `num_epochs` for a smoke test; use the full setting for result reproduction.


In [ ]:
model_resnet = ResNetClassifier(num_classes=num_classes_2d, pretrained=True).to(device)

train_losses_resnet, val_losses_resnet, val_accs_resnet = train_cnn2d(
    model_resnet,
    train_loader_2d_aug,
    val_loader_2d,
    device,
    num_classes_2d,
    num_epochs=100,
    lam_scl_weight=0.1,
    use_swscl=True,
    use_tta=True,
    tta_transform=tta_transform,
    similarity_matrix_fn=custom_similarity_matrix,
    save_path="best_resnet_supcon_model.pth",
    learning_rate=0.001,
)


## Evaluate Saved Checkpoint

Evaluation reloads the best validation checkpoint before reporting the held-out test classification report and confusion matrix.


In [ ]:
model_resnet = ResNetClassifier(num_classes=num_classes_2d, pretrained=False).to(device)
model_resnet.load_state_dict(torch.load("best_resnet_supcon_model.pth", map_location=device))
evaluate(model_resnet, test_loader_2d, device, label_encoder)


## Inspect Representations

The ResNet embedding returned by `get_embeddings` is one 512-dimensional vector per image, matching the representation dimensionality used by the lightweight CNN helper API.


In [ ]:
model_resnet.eval()
images, labels = next(iter(test_loader_2d))
images = images.to(device)

with torch.no_grad():
    embeddings = model_resnet.get_embeddings(images)

print(f"Batch shape: {tuple(images.shape)}")
print(f"Representation shape: {tuple(embeddings.shape)}")
